In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated,Literal
from pydantic import BaseModel, Field
import operator#reducer fn


In [2]:
import httpx

try:
    r = httpx.get(
        "https://api.groq.com/openai/v1/models",
        timeout=15
    )
    print("HTTP status:", r.status_code)
except Exception as e:
    print(type(e).__name__, repr(e))

ConnectError ConnectError('[Errno 8] nodename nor servname provided, or not known')


In [3]:
load_dotenv()

True

In [4]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [5]:
class SentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')


In [6]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [7]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2=model.with_structured_output(DiagnosisSchema)

In [8]:
prompt = "What is the sentiment of the following review: The software is  too good"
result = structured_model.invoke(prompt).sentiment
print(result)

APIConnectionError: Connection error.

In [ ]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [ ]:
def find_sentiment(state: ReviewState):
    prompt = f"For the following review, find the sentiment:\n{state['review']}"

    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}
def check_sentiment(state: ReviewState)->Literal["positive_response","negative_response"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'

    else:
        return 'run_diagnosis'
def positive_response(state:ReviewState):
    prompt=f"""Write a warm thank-you mesagge in response to review:\n\n"{state['review']}\"\ng
    also kindly ask the user to leave feedback on the website"""
    response=model.invoke(prompt)
    return {'response':response}
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}#pydantic opbejct dump used

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']#dict

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

result=workflow.in voke(intial_state)
print(result)

KeyError: 'run_diagnosis'

In [ ]:
(